# Least Squares Line Fit

Chapter 3: Subspaces and Orthogonality · *Mathematics for AI and Machine Learning* by Fuheng Wu

Fit y=ax+b by minimizing squared residuals and visualize errors.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-3/least-squares-line/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

# The demo's data: true line y = a x + b plus a fixed noise pattern
a_true, b_true, noise = 0.8, 0.2, 0.25
x = torch.tensor([-2.0, -1.0, 0.0, 1.0, 2.0, 3.0], dtype=torch.float64)
pattern = torch.tensor([0.2, -0.4, 0.1, 0.5, -0.3, 0.25], dtype=torch.float64)
y = a_true * x + b_true + noise * pattern

# ── Design matrix A = [x, 1]; minimize ||A [a, b]^T - y||^2 ────────────────
A = torch.stack([x, torch.ones_like(x)], dim=1)

# 1) Normal equations  (A^T A) w = A^T y
w_normal = torch.linalg.solve(A.T @ A, A.T @ y)
# 2) Library least squares (QR/SVD based, better conditioned)
w_lstsq = torch.linalg.lstsq(A, y.unsqueeze(1)).solution.squeeze(1)

print("fitted [a, b] (normal eq.) =", w_normal.tolist())
print("fitted [a, b] (lstsq)      =", w_lstsq.tolist())

residuals = y - A @ w_lstsq
print("residuals =", [round(r, 4) for r in residuals.tolist()])
print("sum of squared residuals =", residuals.pow(2).sum().item())

# The residual is orthogonal to every column of A (that is what "least" means)
print("A^T r (should be ~0) =", (A.T @ residuals).tolist())

# ── Same fit by gradient descent with autograd ──────────────────────────────
w = torch.zeros(2, dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([w], lr=0.05)
for _ in range(2000):
    opt.zero_grad()
    loss = ((A @ w - y) ** 2).mean()
    loss.backward()
    opt.step()
print("\ngradient descent [a, b] =", [round(t, 6) for t in w.tolist()])

fitted [a, b] (normal eq.) = [0.8067857142857143, 0.21119047619047626]
fitted [a, b] (lstsq)      = [0.8067857142857142, 0.2111904761904762]
residuals = [0.0524, -0.1044, 0.0138, 0.107, -0.0998, 0.031]
sum of squared residuals = 0.03619940476190478
A^T r (should be ~0) = [3.4416913763379853e-15, 8.604228440844963e-16]



gradient descent [a, b] = [0.806786, 0.21119]
